In [ ]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 데이터 불러오기

In [ ]:
# 데이터 불러오기
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

> 다양한 개인적 특성을 바탕으로 한 데이터를 활용하여 ***소득 수준을 예측***

## train

In [ ]:
train.head()

In [ ]:
train.info()
# 결측치 없음 !

In [ ]:
train.describe()

> ***Feature Info***

* 한 사람에 관련된 다양한 사회적, 경제적 정보
* ID : 학습 데이터 고유 ID
* Age : 나이
* Gender : 성별
* Education_Status : 교육 수준
* Employment_Status : 고용 상태
* Working_Week (Yearly) : 연간 근로 주 수
* Industry_Status : 산업 상태
* Occupation_Status : 직업 상태
* Race : 인종
* Hispanic_Origin : 히스패닉 출신 여부 (히스패닉 - 스페인어를 사용하는 문화권 출신)
* Martial_Status : 결혼 상태
* Household_Status : 가구 현황
* Household_summary : 가구 개요
* Citizenship : 시민권
* Birth_Country : 출생 국가
* Birth_Country (Father) : 아버지의 출생 국가
* Birth_Country (Mother) : 어머니의 출생 국가
* Tax_Status : 세금 지불 상태
* Gains : 소득금액
* Losses : 손실금액
* Divdends : 배당금
* Incom_Status : 소득 상태
* ***Income*** : 예측 목표, 1시간 단위의 소득 ($)

In [ ]:
# 불필요한 칼럼 제거
train.drop(columns=['ID'], inplace=True)
train

* 히스패닉은 주로 미국에서 사용되는 용어 -> 이 데이터셋이 미국에서 수집된 데이터라고 생각할 수 있음

In [ ]:
# 각 칼럼의 고유값 확인
for column in train.columns:
    value_counts = train[column].value_counts()
    print(f"{value_counts}\n")

## test

In [ ]:
test.info()
# Household_Status 에 결측치 하나 존재 -> test 데이터 셋의 통계 값 활용

In [ ]:
test.describe()

# EDA

## 전체 칼럼 확인

In [ ]:
# 수치형 칼럼에 대한 박스플롯
num_cols = ['Age', 'Working_Week (Yearly)', 'Gains', 'Losses', 'Dividends', 'Income']
plt.figure(figsize=(15, 6))
for i, feature in enumerate(num_cols):
    plt.subplot(2, 3, i+1)
    sns.boxplot(x=train[feature])
    plt.title(f'Box Plot of {feature}')
plt.tight_layout()
plt.show()

In [ ]:
# 수치형 칼럼 간의 히트맵 시각화
correlation_matrix = train[num_cols].corr()
plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='YlGnBu', fmt=".4f", linewidths=0.5)
plt.title('Correlation of Num Cols')
plt.show()

* Income 과 비교적 상관관계가 있는 것은 'Working_Week (Yearly)'

## *Income* (예측 목표)

* 1시간 단위의 소득, 단위는 달러
* Income이 0일 경우, 근로소득이 없다

In [ ]:
# 소득의 기초통계량 확인
train['Income'].describe().reset_index()

In [ ]:
# 소득의 분포 시각화
plt.figure(figsize=(10, 4))

# Histogram
plt.subplot(1, 2, 1)
sns.histplot(train['Income'], kde=True)
plt.title('Histogram')
plt.xlabel('Income')
plt.ylabel('Frequency')

# Box PLot
plt.subplot(1, 2, 2)
sns.boxplot(x=train['Income'])
plt.title('Box Plot')
plt.xlabel('Income')

plt.tight_layout()
plt.show()

In [ ]:
# 소득의 각 고유값 확인
train['Income'].value_counts()

> 소득이 0인게 많다 -> 0에 집중해보기 !

In [ ]:
# 구간을 5개로 나눠서 값 개수 확인
pd.cut(train['Income'], bins=5, labels=['0-2000', '2000-4000','4000-6000',
                                        '6000-8000', '8000-10000']).value_counts().sort_index()

## Age

In [ ]:
# Age 칼럼의 분포 확인
plt.figure(figsize=(8, 6))
plt.hist(train['Age'], bins=20, color='lightblue', edgecolor='gray')
plt.title('Distribution of Age')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

* 20대부터 소득이 급격히 늘어남 -> 성인 & 사회 진출
* 20대 ~ 40대에 소득이 많은 편 -> 사회 생활
* 60대부터 소득이 급격히 줄어듬 -> 은퇴시기

In [ ]:
# 나이에 따른 소득 확인하기
plt.figure(figsize=(12, 8))
sns.boxplot(x='Age', y='Income', data=train, palette='Set3')
plt.title('Income by Age')
plt.xlabel('Age')
plt.ylabel('Income')
plt.xticks(rotation=90)
plt.show()

In [ ]:
train[(train['Age'] == 0) & (train['Income'] != 0)]
# 0세인 데이터의 소득은 모두 0

In [ ]:
train[(1 <= train['Age']) & (train['Age'] <= 14) & (train['Income'] != 0)]
# 1 ~ 14세 중 소득이 0이 아닌 데이터 없음

> 0 ~ 14세는 소득이 분명히 0일 것이라 판단 !

In [ ]:
train = train[~((train['Age'] >= 0) & (train['Age'] <= 14))]
train.shape

## Gender

* M : 남자 / F : 여자

In [ ]:
# 성별에 따른 소득의 평균값 확인
train.groupby('Gender')['Income'].mean().reset_index()

In [ ]:
train['Gender'].value_counts()
# 성비는 거의 반반

In [ ]:
# 소득이 0인 데이터의 성별 확인
train[train['Income'] == 0]['Gender'].value_counts()

In [ ]:
# 성별에 따른 소득 분포 확인
plt.figure(figsize=(8, 6))
sns.boxplot(x='Gender', y='Income', data=train, hue='Gender')
plt.title('Income by Gender')
plt.xlabel('Gender')
plt.ylabel('Income')
plt.show()

In [ ]:
# 카이제곱 검정 수행
from scipy.stats import chi2_contingency
cross_tab = pd.crosstab(train['Gender'], train['Income'])
chi2, p_value, dof, expected = chi2_contingency(cross_tab)
print("Chi-square Test Statistic:", chi2)
print("p-value:", p_value)

* 카이제곱 검정을 통해 'Gender'는 'Income'과 통계적으로 유의미함을 확인

## Education_Status

* Doctorate degree: 박사 학위
* Professional degree: 전문 학위
* Masters degree: 석사 학위
* Bachelors degree: 학사 학위
* Associates degree (Academic): 전문학사 학위 (학문적)
* Associates degree (Vocational): 전문학사 학위 (전문 직업)
* College: 대학 진학
* High graduate: 고등학교 졸업 (한 상태)
* High Senior: 고등학교 졸업생 (12학년)
* High Junior: 고등학교 3학년 (11학년)
* High Sophomore: 고등학교 2학년 (10학년)
* High Freshman: 고등학교 1학년 (9학년)
* Middle (7-8): 중학교 (7-8학년)
* Elementary (5-6): 초등학교 (5-6학년)
* Elementary (1-4): 초등학교 (1-4학년)
* Kindergarten: 유치원
* Children: 아이들

In [ ]:
train.groupby('Education_Status')['Income'].mean().reset_index()

> Childern 의 소득 평균 0 -> Childern은 분명히 소득 0 -> 삭제해도 될듯

In [ ]:
# Education_Status 칼럼의 분포 확인
education_income_mean = train.groupby('Education_Status')['Income'].mean().sort_values()
plt.figure(figsize=(10, 6))
plt.barh(education_income_mean.index, education_income_mean.values, color='skyblue')
plt.xlabel('Mean Income')
plt.ylabel('Education Status')
plt.title('Mean Income by Education Status')
plt.show()

In [ ]:
education_status = 'Education_Status'

summary_stats = pd.DataFrame()

for value in train[education_status].unique():
    subset = train[train[education_status] == value]
    stats = subset['Income'].describe().reset_index()
    stats.rename(columns={'index': 'Statistic'}, inplace=True)
    stats['Education_Status'] = value
    summary_stats = pd.concat([summary_stats, stats], ignore_index=True)

print(summary_stats.pivot(index='Education_Status', columns='Statistic', values='Income'))

In [ ]:
# 학력 상태별로 그룹화하기
def group_education(status):
    if status in [
        'Kindergarten',
        'Elementary (1-4)',
        'Elementary (5-6)',
        'Middle (7-8)'
    ]:
        return 'Teen'
    elif status == 'High Freshman':
        return 'High Freshman'
    elif status in [
        'High Sophomore',
        'High Junior',
        'High Senior'
    ]:
        return 'High School'
    elif status in [
        'High graduate',
        'College'
    ]:
        return 'Graduated'
    elif status == 'Associates degree (Vocational)':
        return 'Associates degree (Vocational)'
    elif status in [
        'Associates degree (Academic)',
        'Bachelors degree',
        'Masters degree',
        'Professional degree'
    ]:
        return 'Degrees'
    elif status == 'Doctorate degree':
        return 'Doctorate degree'
    elif status == 'Children':
        return 'Children'
    else:
        return 'Other'

def apply_Education(train, test):
    train['Education_Status'] = train['Education_Status'].apply(group_education)
    test['Education_Status'] = test['Education_Status'].apply(group_education)
    return train, test

train, test = apply_Education(train, test)
train['Education_Status'].value_counts()

## Employment_Status

* Children or Armed Forces: 어린이 or 군인 (일자리 X)
* Full-Time: 전일제
* Not Working: 고용 X
* Choice Part-Time: 선택적 파트타임
* Part-Time (Usually Part-Time): 파트타임 (보통 파트타임)
* Seeking Full-Time: 전일제를 찾는 중
* Part-Time (Usually Full-Time): 파트타임 (보통 전일제)
* Seeking Part-Time: 파트타임을 찾는 중

In [ ]:
train.groupby('Employment_Status')['Income'].mean().reset_index()

* Not Working 의 소득 평균 0 -> 분명히 소득이 0 일것이라 판단

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='Income', y='Employment_Status', data=train, palette='Set3')
plt.title('Income by Employment Status')
plt.xlabel('Income')
plt.ylabel('Employment Status')
plt.show()

In [ ]:
# 고용 상태 별 Working_Week 의 평균 확인
train.groupby('Employment_Status')['Working_Week (Yearly)'].mean().reset_index()

In [ ]:
from scipy.stats import f_oneway, kruskal

# 각 Employment_Status 클래스에 대해 소득 데이터 추출
children_or_armed_forces_income = train[train['Employment_Status'].isin(['Children or Armed Forces'])]['Income']
full_time_income = train[train['Employment_Status'].isin(['Full-Time'])]['Income']
choice_part_time_income = train[train['Employment_Status'].isin(['Choice Part-Time'])]['Income']
part_time_usually_part_time_income = train[train['Employment_Status'].isin(['Part-Time (Usually Part-Time)'])]['Income']
part_time_usually_full_time_income = train[train['Employment_Status'].isin(['Part-Time (Usually Full-Time)'])]['Income']

# ANOVA 수행
anova_result = f_oneway(children_or_armed_forces_income, full_time_income, choice_part_time_income,
                        part_time_usually_part_time_income,
                        part_time_usually_full_time_income)

# Kruskal-Wallis 테스트 수행
kruskal_result = kruskal(children_or_armed_forces_income, full_time_income, choice_part_time_income,
                          part_time_usually_part_time_income, part_time_usually_full_time_income)

# 결과 출력
print("ANOVA 결과:")
print("F-statistic:", anova_result.statistic)
print("p-value:", anova_result.pvalue)
print("\nKruskal-Wallis 테스트 결과:")
print("H-statistic:", kruskal_result.statistic)
print("p-value:", kruskal_result.pvalue)

In [ ]:
# Seeking 중에서 소득이 0인 데이터 확인
train[(train['Employment_Status'] == 'Seeking Full-Time') |
      (train['Employment_Status'] == 'Seeking Part-Time') &
      (train['Income'] == 0)]

> 구직중인데 소득이 0 인건 분명함 -> train에서 삭제

In [ ]:
# Employment_Status의 범주를 그룹화하는 함수 정의
def Employment_assign_cluster(status):
    if status == 'Children or Armed Forces':
        return 'Children or Armed Forces'
    else:
        return 'Not Children or Armed Forces'

def apply_Employment(train, test):
    train['Employment_Status'] = train['Employment_Status'].apply(Employment_assign_cluster)
    test['Employment_Status'] = test['Employment_Status'].apply(Employment_assign_cluster)
    return train, test

train, test = apply_Employment(train, test)
train['Employment_Status'].value_counts()

## Working_Week (Yearly)

In [ ]:
# 기초통계량 확인
train['Working_Week (Yearly)'].describe().reset_index()

In [ ]:
# 근로 주수 별 소득 분포 확인
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Working_Week (Yearly)', y='Income', color='green', data=train)
plt.title('Income vs Working Week')
plt.xlabel('Working Week (Yearly)')
plt.ylabel('Income')
plt.show()

> 52시간으로 가장 많이 일 하는 사람들의 소득이 높게 분포

In [ ]:
# 주에 52시간 일하는 사람 중 소득이 0인 데이터 확인
train[(train['Working_Week (Yearly)'] == 52) & (train['Income'] == 0)]

In [ ]:
train[(train['Working_Week (Yearly)'] == 52) & (train['Income'] == 0)]['Employment_Status'].value_counts()

In [ ]:
# 주에 일하는 시간은 존재 & 소득은 0인 데이터 확인
train[(train['Working_Week (Yearly)'] > 0) & (train['Income'] == 0)]

In [ ]:
train[(train['Working_Week (Yearly)'] == 0) & (train['Income'] == 0)]

In [ ]:
# Working_Week의 범주로 그룹화하는 함수 정의
def Working_Week_assign_cluster(label):
    if label < 4:
        return 'under 30'
    elif label < 6:
        return 'under 51'
    else:
        return '52'

def apply_Working_Week(train, test):
    train['Working_Week (Yearly)'] = train['Working_Week (Yearly)'].apply(Working_Week_assign_cluster)
    test['Working_Week (Yearly)'] = test['Working_Week (Yearly)'].apply(Working_Week_assign_cluster)
    return train, test

# Working_Week의 범주로 그룹화하는 함수 적용
train, test = apply_Working_Week(train, test)
train['Working_Week (Yearly)'].value_counts()

## Household_Status

In [ ]:
# 가구 개요에 따른 소득 평균 확인
train.groupby('Household_Status')['Income'].mean().reset_index()

In [ ]:
# 카이제곱 검정 수행
cross_tab = pd.crosstab(train['Household_Status'], train['Income'])
chi2, p_value, dof, expected = chi2_contingency(cross_tab)

print("Chi-square Test Statistic:", chi2)
print("p-value:", p_value)

> Household_Summary는 Household_Status의 내용을 기반으로 요약한 것
>
> & Household_Status 은 카이제곱 검정을 통해 통계적으로 유의미한 관련성이 없음을 확인
>
> --> Household_Summary 칼럼만 활용  !

In [ ]:
# 카이제곱 검정 수행
cross_tab = pd.crosstab(train['Household_Summary'], train['Income'])
chi2, p_value, dof, expected = chi2_contingency(cross_tab)

print("Chi-square Test Statistic:", chi2)
print("p-value:", p_value)

* Household_Status 칼럼 drop 전 Household_Summary 의 p-value 는 0.55

In [ ]:
train_drop_hs = train.drop('Household_Status', axis=1)

In [ ]:
# 카이제곱 검정 수행
cross_tab = pd.crosstab(train_drop_hs['Household_Summary'], train_drop_hs['Income'])
chi2, p_value, dof, expected = chi2_contingency(cross_tab)

print("Chi-square Test Statistic:", chi2)
print("p-value:", p_value)

> Household_Status 칼럼 drop 전 Household_Summary 의 p-value 와 동일
>
> --> Household_Status 를 삭제 하더라도 Income 영향 X (유의미한 관계 X)
>
> --> Household_Status drop !!

In [ ]:
train = train_drop_hs

In [ ]:
train.nunique()

## Household_Summary

* Householder: 주택 소유주
* Spouse of householder: 주택 소유주의 배우자
* Child under 18 never married: 미혼 18세 미만 자녀
* Child 18 or older: 18세 이상 자녀
* Nonrelative of householder: 주택 소유주의 비가족
* Other relative of householder: 주택 소유주의 다른 가족 구성원
* Group Quarters- Secondary individual: 그룹 주거시설 - 보조 개인
* Child under 18 ever married: 결혼 경험이 있는 18세 미만 자녀

In [ ]:
# 가구 개요에 따른 소득 평균 확인
train.groupby('Household_Summary')['Income'].mean().reset_index()

In [ ]:
train['Household_Summary'].value_counts()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='Income', y='Household_Summary', data=train)
plt.title('Income Distribution by Household Summary')
plt.xlabel('Income')
plt.ylabel('Household Summary')
plt.show()

In [ ]:
# 가구 상태별로 그룹화하기
def group_household_status(status):
    if status in ['Child under 18 never married', 'Child under 18 ever married']:
        return 'Child under 18'
    elif status in ['Spouse of householder', 'Householder']:
        return 'Household & Spouse'
    else:
        return status


def apply_Household(train, test):
    train['Household_Summary'] = train['Household_Summary'].apply(group_household_status)
    test['Household_Summary'] = test['Household_Summary'].apply(group_household_status)
    return train, test

train, test = apply_Household(train, test)
train['Household_Summary'].value_counts()

In [ ]:
from scipy.stats import f_oneway

# Household_Summary 열을 기준으로 데이터를 그룹화하여 각 그룹에 대한 소득 데이터 추출
grouped_data = train.groupby('Household_Summary')['Income']

# ANOVA 수행
f_stat, p_val = f_oneway(*[group for name, group in grouped_data])

# 결과 출력
print(f"F-statistic: {f_stat}")
print(f"p-value: {p_val}")

## Citizenship

* Native: 미국인
* Foreign-born (Non-US Citizen): 외국에서 태어남 (미국 시민X)
* Foreign-born (Naturalized US Citizen): 외국에서 태어남 (미국 시민권 취득)
* Native (Born Abroad): 해외에서 태어남 (미국 시민권)
* Native (Born in Puerto Rico or US Outlying): 푸에르토리코 또는 미국 외곽 지역에서 태어난 (미국 시민권)

* 푸에르토리코 (미국의 자치령) -> 미국 시민권
* 미국 외곽 지역 (미국령 섬) -> 미국 시민권

In [ ]:
# 시민권에 따른 소득 평균 확인
train.groupby('Citizenship')['Income'].mean().reset_index()

In [ ]:
train['Citizenship'].value_counts()

In [ ]:
# 카이제곱 검정 수행
cross_tab = pd.crosstab(train['Citizenship'], train['Income'])
chi2, p_value, dof, expected = chi2_contingency(cross_tab)

print("Chi-square Test Statistic:", chi2)
print("p-value:", p_value)

In [ ]:
sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))
sns.boxplot(x='Income', y='Citizenship', data=train, palette='Set3')
plt.title('Income by Citizenship')
plt.xlabel('Income')
plt.ylabel('Citizenship')
plt.show()

In [ ]:
from scipy.stats import f_oneway
import statsmodels.api as sm

# 교육 수준에 따른 소득 데이터 추출
Citizenship = train['Citizenship'].unique()
data = {edu: train[train['Citizenship'] == edu]['Income'] for edu in Citizenship}

# ANOVA 수행
f_stat, p_val = f_oneway(*data.values())

print(f"F-statistic: {f_stat}")
print(f"p-value: {p_val}")

# 사후 검정
tukey_results = sm.stats.multicomp.pairwise_tukeyhsd(train['Income'], train['Citizenship'])

# 결과 출력
print(tukey_results)

* Foreign-born (Non-US Citizen) 이랑 Native (Born in Puerto Rico or US Outlying) 을 묶으려해도 .. 두 값의 큰 연관성 ? 이 없어서 막 할 수가 없다

## Birth_Country

In [ ]:
print(train['Birth_Country'].nunique())
print(train['Birth_Country (Father)'].nunique())
print(train['Birth_Country (Mother)'].nunique())

In [ ]:
train['Birth_Country'].value_counts()

In [ ]:
# 출생국이 Unknown 인 데이터는 삭제
train = train[train['Birth_Country'] != 'Unknown']

In [ ]:
# 출생국에 따른 소득 평균 확인
train.groupby('Birth_Country')['Income'].mean().reset_index()

In [ ]:
# Birth_Country 칼럼에 대한 카이제곱 검정
chi2_birth_country, p_birth_country, _, _ = chi2_contingency(pd.crosstab(train['Birth_Country'], train['Income']))

# Birth_Country (Father) 칼럼에 대한 카이제곱 검정
chi2_birth_country_father, p_birth_country_father, _, _ = chi2_contingency(pd.crosstab(train['Birth_Country (Father)'], train['Income']))

# Birth_Country (Mother) 칼럼에 대한 카이제곱 검정
chi2_birth_country_mother, p_birth_country_mother, _, _ = chi2_contingency(pd.crosstab(train['Birth_Country (Mother)'], train['Income']))

# 결과 출력
print("Birth_Country 칼럼에 대한 카이제곱 검정:")
print("Chi-square Test Statistic:", chi2_birth_country)
print("p-value:", p_birth_country)

print("\nBirth_Country (Father) 칼럼에 대한 카이제곱 검정:")
print("Chi-square Test Statistic:", chi2_birth_country_father)
print("p-value:", p_birth_country_father)

print("\nBirth_Country (Mother) 칼럼에 대한 카이제곱 검정:")
print("Chi-square Test Statistic:", chi2_birth_country_mother)
print("p-value:", p_birth_country_mother)

In [ ]:
from scipy.stats import f_oneway
import statsmodels.api as sm

# 교육 수준에 따른 소득 데이터 추출
Birth_Country = train['Birth_Country'].unique()
data = {edu: train[train['Birth_Country'] == edu]['Income'] for edu in Birth_Country}

# ANOVA 수행
f_stat, p_val = f_oneway(*data.values())

print(f"F-statistic: {f_stat}")
print(f"p-value: {p_val}")

# 사후 검정
tukey_results = sm.stats.multicomp.pairwise_tukeyhsd(train['Income'], train['Birth_Country'])

# 결과 출력
print(tukey_results)

In [ ]:
                    # Columbia                      England   748.8033  0.029    27.6154 1469.9912   True
                    #  England                       Mexico  -602.1557 0.0012 -1091.8656 -112.4459   True
                    #  England                    Nicaragua  -730.1495 0.0421 -1451.3374   -8.9615   True
                    #  England                  South Korea  -759.5771 0.0277  -1488.994  -30.1602   True
                    #   Mexico                  Philippines    435.668    0.0   123.7344  747.6016   True
                    #   Mexico                           US   214.3059    0.0    77.5618  351.0501   True

In [ ]:
# Birth_Country의 범주를 그룹화하는 함수 정의
def Birth_Country_group(status):
    if status == 'US':
        return 'US'
    else:
        return 'Other'

def apply_Birth(train, test):
    train['Birth_Country'] = train['Birth_Country'].apply(Birth_Country_group)
    test['Birth_Country'] = test['Birth_Country'].apply(Birth_Country_group)
    return train, test

train, test = apply_Birth(train, test)
train['Birth_Country'].value_counts()

In [ ]:
from scipy.stats import f_oneway
import statsmodels.api as sm

# 교육 수준에 따른 소득 데이터 추출
Birth_Country = train['Birth_Country '].unique()
data = {edu: train[train['Birth_Country '] == edu]['Income'] for edu in Birth_Country }

# ANOVA 수행
f_stat, p_val = f_oneway(*data.values())

print(f"F-statistic: {f_stat}")
print(f"p-value: {p_val}")

# 사후 검정
tukey_results = sm.stats.multicomp.pairwise_tukeyhsd(train['Income'], train['Birth_Country '])

# 결과 출력
print(tukey_results)

> Panama 는 4개의 데이터가 모두 0 -> 데이터 오류라 판단 -> 이상치로 삭제

> Trinadad&Tobago 는 8개의 데이터 평균이 1115로 가장 높음 -> 이상치로 판단 -> 삭제

In [ ]:
# 출생국(아버지)에 따른 소득 평균 확인
train.groupby('Birth_Country (Father)')['Income'].mean().reset_index()

In [ ]:
train['Birth_Country (Father)'].value_counts()

In [ ]:
# 출생국(어머니)에 따른 소득 평균 확인
train.groupby('Birth_Country (Mother)')['Income'].mean().reset_index()

> Birth_Country (Mother) & Birth_Country (Father) 에도 동일하게 적용 (그룹화)

In [ ]:
# Birth_Country (Father)의 범주를 그룹화하는 함수 정의
def Birth_Country_group(status):
    if status == 'US':
        return 'US'
    else:
        return 'Other'

def apply_Birth(train, test):
    train['Birth_Country (Father)'] = train['Birth_Country (Father)'].apply(Birth_Country_group)
    test['Birth_Country (Father)'] = test['Birth_Country (Father)'].apply(Birth_Country_group)
    return train, test

train, test = apply_Birth(train, test)
train['Birth_Country (Father)'].value_counts()

In [ ]:
# Birth_Country (Mother)의 범주를 그룹화하는 함수 정의
def Birth_Country_group(status):
    if status == 'US':
        return 'US'
    else:
        return 'Other'

def apply_Birth(train, test):
    train['Birth_Country (Mother)'] = train['Birth_Country (Mother)'].apply(Birth_Country_group)
    test['Birth_Country (Mother)'] = test['Birth_Country (Mother)'].apply(Birth_Country_group)
    return train, test

train, test = apply_Birth(train, test)
train['Birth_Country (Mother)'].value_counts()

In [ ]:
print(train.groupby('Birth_Country').size())
print(train.groupby('Birth_Country (Father)').size())
print(train.groupby('Birth_Country (Mother)').size())

# Data Preprocessing

## 이상치 처리

* Age 칼럼에서는 0~14세의 소득이 모두 0으로 나타남 & 14세까지가 중학생이기때문에 소득이 0일 것으로 판단
> ***0~14세의 데이터는 train에서 삭제 !***

* Education_Status 칼럼에서 Children 의 소득 평균이 0
> ***Childern 데이터는 train에서 삭제 !***

* Employment_Status 칼럼에서 Not Working 의 소득 평균이 0
> ***Not Working 데이터는 train에서 삭제 !***

In [ ]:
train.shape

In [ ]:
# 소득이 분명히 0인 데이터 삭제
train = train[~((train['Age'] >= 0) & (train['Age'] <= 14))]
train = train[train['Education_Status'] != 'Children']
train = train[train['Employment_Status'] != ('Not Working', 'Seeking Full-Time', 'Seeking Part-Time')]
train = train[train['Industry_Status'] != 'Not in universe or children']
train = train[train['Occupation_Status'] != 'Unknown']
train.shape